In [ ]:
import pandas as pd
from typing import List

%cd ..
from utils.estimators import DistributionUE, LabelProbability, SequenceProbability, DistributionOfProbUE, EmbeddingsBasedUE
from utils.metrics import ECE, Brier, ROCAUC
from utils.analysis_utils import *
from utils.statistical_tests import *
%cd ..

/mnt/c/Users/P096811/OneDrive - Amsterdam UMC/Bureaublad/uncertainty/code/Uncertainty_MedVQA_scripts


In [ ]:
N = 210 
M = 8
K = 4  
S = 10  
top_p = 0.9
temperature = 0.6
replace_token = None
prompt_template = "default"
DATE = None
per = None
filter_by = {}
filter_samples = None

metrics = [
    {
        "ue_method": LabelProbability(),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": SequenceProbability(aggregate="mean"),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": SequenceProbability(aggregate="max"),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": DistributionUE("SC", normalize_options=False),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": DistributionUE("SE", normalize_options=False),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": DistributionOfProbUE(normalize_options=False),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": EmbeddingsBasedUE(method_name="EE"),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": EmbeddingsBasedUE(method_name="RDS"),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
]

os.makedirs("outputs/GMAI-MMBench/plots", exist_ok=True)

In [ ]:
def compute_estimator_outputs(df, metrics: List[dict]) -> pd.DataFrame:
    df = df[df["extracted_model_response_mapped"].notna()]
    df = df[df['extracted_model_response_mapped'] != "[]"]
    stats = calculate_sample_level_stats(df, S=10)

    result = {}
    for group in metrics:
        estimator = group["ue_method"]
        estimator_name = str(estimator)
        estimator_outputs = estimator(stats)
        print(f"Computing outputs for estimator: {estimator_name}, len: {len(estimator_outputs)}")
        estimator_outputs = np.asarray(estimator_outputs, dtype=float)
        estimator_outputs = min_max_normalize(model, estimator_name, estimator_outputs)
        result[estimator_name] = estimator_outputs

    result["shuffle_idx"] = stats["shuffle_idx"]
    result["sample_idx"] = stats["sample_idx"]
    results_df = pd.DataFrame(result)

    return results_df

def get_full_stats(model_name, settings):
    df_all = load_data_direct(model, f"{base_folder}/N_210-K_4-M_8/{model_name}-default-temp_0.6-topp_0.9-S_10/extracted.tsv", 
            replace_token=replace_token, filter=None)
    results_df = compute_estimator_outputs(df_all, metrics)

    def load_ids(setting):
        filter_df = pd.read_csv(f"{base_folder}/N_210-K_4-M_8/{model_name}-default-temp_0.6-topp_0.9-S_10/{setting}.csv")
        return filter_df

    samples_ids = {
        setting: load_ids(setting) for setting in settings    
    }

    for setting in settings:
        results_df[setting] = False
        ids = list(samples_ids[setting].itertuples(index=False, name=None))
        for sample_idx, shuffle_idx in ids:
            results_df.loc[(results_df["sample_idx"] == sample_idx) & (results_df["shuffle_idx"] == shuffle_idx), setting] = True

    return results_df

In [ ]:
model = "Qwen2.5-VL-72B-Instruct"
base_folder = "outputs/GMAI-MMBench"
ue_methods = [metric.get("ue_method") for metric in metrics if metric.get("ue_method") is not None]

settings = ["plus_none_c_to_c", "plus_none_c_to_w",
            "correct_to_none_c_to_c", "correct_to_none_c_to_w",
            "originally_correct", "originally_wrong"]

results_df = get_full_stats(model, settings)
results_df_originally_correct = results_df[results_df["originally_correct"] == True]
results_df_originally_wrong = results_df[results_df["originally_correct"] == False]

results_df_originally_correct.head()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

def compute_metrics(results, ue_methods, target_col='correct_to_none_c_to_w'):
    """
    Compute AUROC, AUPRC, selective risk, and risk-coverage curves for each UE method.

    Args:
        results (pd.DataFrame): DataFrame with UE method columns and target_col.
        ue_methods (list): List of column names for UE methods.
        target_col (str): Name of the binary target column.

    Returns:
        dict: Dictionary with metrics and curves for each UE method.
    """
    # Add the baselines to the results_df
    majority_class = results[target_col].mode()[0]
    results['majority_class_baseline'] = majority_class
    results['random_baseline'] = np.random.choice([0, 1], size=len(results))
    results['random_prior_baseline'] = np.random.choice([0, 1], size=len(results), p=[1 - results[target_col].mean(), results[target_col].mean()])
    results['random_ue_baseline'] = np.random.rand(len(results))  # Random uncertainty estimates between 0 and 1
    
    metrics = {}

    for method in ue_methods+['majority_class_baseline', 'random_baseline', 'random_prior_baseline', 'random_ue_baseline']:
        ue_name = str(method)
        filtered_df = results[[ue_name, target_col]].dropna()
        ue_values = filtered_df[ue_name].values
        y_true = filtered_df[target_col].values

        # AUROC
        auroc = roc_auc_score(y_true, ue_values)
        # AUPRC
        auprc = average_precision_score(y_true, ue_values)

        # fixed length thresholds
        thresholds = np.linspace(0, 1.01, 102)
        selective_risks = []
        threshold_values = []
        coverages = []

        for threshold in thresholds:
            answered = ue_values < threshold
            coverage = answered.mean()

            if coverage > 0:
                selective_risk = y_true[answered].mean()   # if 1 = fragile
            else:
                selective_risk = 0

            selective_risks.append(selective_risk)
            threshold_values.append(threshold)
            coverages.append(coverage)

        # Store results
        metrics[ue_name] = {
            'AUROC': auroc,
            'AUPRC': auprc,
            'thresholds': threshold_values,
            'selective_risks': selective_risks,
            'coverages': coverages,
        }

    return metrics

In [126]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

def compute_metrics(results, ue_methods, target_col='correct_to_none_c_to_w'):
    """
    Compute AUROC, AUPRC, and precision-recall curves at quantile thresholds for each UE method.

    Args:
        results (pd.DataFrame): DataFrame with UE method columns and target_col.
        ue_methods (list): List of column names for UE methods.
        target_col (str): Name of the binary target column.

    Returns:
        dict: Dictionary with metrics and curves for each UE method.
    """
    # Add the baselines to the results_df
    majority_class = results[target_col].mode()[0]
    results['majority_class_baseline'] = majority_class.astype(int)
    results['random_baseline'] = np.random.choice([0, 1], size=len(results)).astype(int)
    results['random_prior_baseline'] = np.random.choice([0, 1], size=len(results), p=[1 - results[target_col].mean(), results[target_col].mean()]).astype(int)
    results['random_ue_baseline'] = np.random.rand(len(results)).astype(int)  # Random uncertainty estimates between 0 and 1

    metrics = {}

    for method in ue_methods+['majority_class_baseline', 'random_baseline', 'random_prior_baseline', 'random_ue_baseline']:
        ue_name = str(method)
        # print(f"Computing metrics for UE method: {ue_name}")
        filtered_df = results[[ue_name, target_col]].dropna()
        ue_values = filtered_df[ue_name].values
        y_true = filtered_df[target_col].values

        # AUROC
        auroc = roc_auc_score(y_true, ue_values)
        # AUPRC
        auprc = average_precision_score(y_true, ue_values)

        # Quantile thresholds: 0%, 10%, 20%, ..., 100%
        quantiles = np.arange(0, 1.01, 0.1)
        precisions = []
        recalls = []
        
        for q in quantiles:
            # Threshold at this quantile: samples with score >= threshold are considered "selected"
            threshold = np.quantile(ue_values, q)
            
            if q == 0.0:
                # Keep ALL samples (even if ue=0)
                selected = np.ones(len(ue_values), dtype=bool)
            elif q == 1.0:
                # Exclude ALL samples (even if ue=0)
                selected = np.zeros(len(ue_values), dtype=bool)
            else:
                # Sort by uncertainty (ascending), keep the bottom n_keep samples
                # These are the samples with LOWEST uncertainty
                selected = ue_values >= threshold

            # Precision: among selected samples, what ratio has target=1?
            if selected.sum() > 0:
                precision = y_true[selected].mean()
            else:
                precision = 1.0  # Undefined, set to 1 conventionally
            
            # Recall: among all positive samples, what ratio was selected?
            if y_true.sum() > 0:
                recall = y_true[selected].sum() / y_true.sum()
            else:
                recall = 0

            precisions.append(precision)
            recalls.append(recall)

        # Store results
        metrics[ue_name] = {
            'AUROC': auroc,
            'AUPRC': auprc,
            'quantiles': quantiles,
            'precisions': precisions,
            'recalls': recalls,
        }

    return metrics

In [127]:
metrics_results_plus_nota = compute_metrics(results_df_originally_correct, ue_methods, target_col='plus_none_c_to_w')
metrics_results_correct_to_nota = compute_metrics(results_df_originally_correct, ue_methods, target_col='correct_to_none_c_to_w')

positive = len(results_df[results_df["plus_none_c_to_w"]])
negative = len(results_df[results_df["plus_none_c_to_c"]])
chance_plus_nota = positive / (positive + negative)
positive = len(results_df[results_df["correct_to_none_c_to_w"]])
negative = len(results_df[results_df["correct_to_none_c_to_c"]])
chance_correct_to_nota = positive / (positive + negative)

for i, ue_name in enumerate(metrics_results_plus_nota.keys()):
    metric_data_plus_nota = metrics_results_plus_nota[ue_name]
    metric_data_correct_to_nota = metrics_results_correct_to_nota[ue_name]
    ue_name = ue_name.replace("_embeddings-biomedical", "")  # Escape underscores for LaTeX formatting
    print(f"{ue_name} & {metric_data_plus_nota['AUROC']:.3f} & {metric_data_plus_nota['AUPRC']:.3f} & {metric_data_correct_to_nota['AUROC']:.3f} & {metric_data_correct_to_nota['AUPRC']:.3f}  \\\\")
    # print(f"| {ue_name} | {metric_data_plus_nota['AUROC']:.3f} | {metric_data_plus_nota['AUPRC']:.3f} | {metric_data_correct_to_nota['AUROC']:.3f} | {metric_data_correct_to_nota['AUPRC']:.3f} |")

# print(f"chance & 0.5 & {chance_plus_nota:.3f} & {chance_correct_to_nota:.3f} & {chance_correct_to_nota:.3f}  \\\\")
# print(f"| chance | 0.5 | {chance_plus_nota:.3f} | 0.5 | {chance_correct_to_nota:.3f} |")

LabelNLL & 0.798 & 0.616 & 0.648 & 0.675  \\
ANLL & 0.570 & 0.279 & 0.527 & 0.613  \\
MaxNLL & 0.575 & 0.281 & 0.529 & 0.614  \\
SC & 0.771 & 0.547 & 0.593 & 0.652  \\
SE & 0.773 & 0.562 & 0.591 & 0.648  \\
PRO & 0.753 & 0.479 & 0.569 & 0.625  \\
EE & 0.713 & 0.445 & 0.577 & 0.672  \\
RDS & 0.717 & 0.450 & 0.575 & 0.666  \\
majority_class_baseline & 0.500 & 0.251 & 0.500 & 0.600  \\
random_baseline & 0.503 & 0.252 & 0.514 & 0.607  \\
random_prior_baseline & 0.494 & 0.248 & 0.516 & 0.607  \\
random_ue_baseline & 0.500 & 0.251 & 0.500 & 0.600  \\


/tmp/ipykernel_22359/3320502093.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_22359/3320502093.py:18: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_22359/3320502093.py:19: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykerne